In [1]:
from __future__ import annotations

import os
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "HardwareNamedVector"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

hardware = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=[
        wvc.config.Configure.Vectors.text2vec_openai(
            name="title_vector",
            source_properties=["title"],
            model="text-embedding-3-small",
        ),
        wvc.config.Configure.Vectors.text2vec_openai(
            name="description_vector",
            source_properties=["description"],
            model="text-embedding-3-small",
        ),
    ],
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="description",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="component",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="use_case",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: HardwareNamedVector


In [5]:
hardware_objects = [
    {
        "title": "NVIDIA RTX 4000 Ada",
        "description": "A professional GPU with large VRAM, useful for machine learning, rendering, CUDA workloads and local AI experiments.",
        "component": "GPU",
        "use_case": "AI workstation",
    },
    {
        "title": "AMD Ryzen 9 Processor",
        "description": "A high core count CPU useful for compiling code, virtualization, multitasking and running development environments.",
        "component": "CPU",
        "use_case": "development workstation",
    },
    {
        "title": "Samsung 990 PRO NVMe SSD",
        "description": "A very fast NVMe storage drive useful for operating systems, datasets, Docker images, virtual machines and large project files.",
        "component": "Storage",
        "use_case": "fast storage",
    },
    {
        "title": "64 GB DDR5 RAM Kit",
        "description": "Large system memory helps with Docker containers, virtual machines, data science notebooks and large datasets.",
        "component": "RAM",
        "use_case": "memory intensive workloads",
    },
    {
        "title": "850W Modular Power Supply",
        "description": "A reliable PSU provides stable power for workstations with powerful GPUs, CPUs and multiple storage drives.",
        "component": "PSU",
        "use_case": "workstation power",
    },
    {
        "title": "PCIe 4.0 Motherboard",
        "description": "A motherboard with modern chipset features, PCIe slots, fast storage connectors and support for future hardware upgrades.",
        "component": "Motherboard",
        "use_case": "hardware expansion",
    },
]

result = hardware.data.insert_many(hardware_objects)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [6]:
response = hardware.query.fetch_objects(
    limit=10,
    return_properties=["title", "component", "use_case"],
)

for obj in response.objects:
    print(obj.uuid)
    print(obj.properties)
    print("-" * 80)

1029ff3e-fbc0-44ca-8d7e-41292303a82b
{'component': 'CPU', 'title': 'AMD Ryzen 9 Processor', 'use_case': 'development workstation'}
--------------------------------------------------------------------------------
31125e8e-f6e0-4576-9fe1-2a6165c62c8c
{'component': 'Motherboard', 'title': 'PCIe 4.0 Motherboard', 'use_case': 'hardware expansion'}
--------------------------------------------------------------------------------
612c4b01-24e2-4fe2-b36a-69f233791a1d
{'component': 'RAM', 'title': '64 GB DDR5 RAM Kit', 'use_case': 'memory intensive workloads'}
--------------------------------------------------------------------------------
7e5bb600-a0eb-4745-a0ed-8b73bdc00d13
{'component': 'PSU', 'title': '850W Modular Power Supply', 'use_case': 'workstation power'}
--------------------------------------------------------------------------------
8db62cf6-54bd-4510-879f-d927488c27d5
{'component': 'Storage', 'title': 'Samsung 990 PRO NVMe SSD', 'use_case': 'fast storage'}
-------------------------

In [7]:
response = hardware.query.fetch_objects(
    limit=1,
    return_properties=["title", "description"],
    include_vector=True,
)

obj = response.objects[0]

print(obj.properties)
print(obj.vector.keys())
print("title_vector length:", len(obj.vector["title_vector"]))
print("description_vector length:", len(obj.vector["description_vector"]))

{'title': 'AMD Ryzen 9 Processor', 'description': 'A high core count CPU useful for compiling code, virtualization, multitasking and running development environments.'}
dict_keys(['description_vector', 'title_vector'])
title_vector length: 1536
description_vector length: 1536


In [8]:
response = hardware.query.near_text(
    query="NVIDIA graphics card",
    target_vector="title_vector",
    limit=3,
    return_properties=["title", "component", "use_case"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print(obj.properties)
    print("-" * 80)

Distance: 0.42690855264663696
{'component': 'GPU', 'title': 'NVIDIA RTX 4000 Ada', 'use_case': 'AI workstation'}
--------------------------------------------------------------------------------
Distance: 0.5365774035453796
{'component': 'Storage', 'title': 'Samsung 990 PRO NVMe SSD', 'use_case': 'fast storage'}
--------------------------------------------------------------------------------
Distance: 0.5603469014167786
{'component': 'Motherboard', 'title': 'PCIe 4.0 Motherboard', 'use_case': 'hardware expansion'}
--------------------------------------------------------------------------------


In [9]:
response = hardware.query.near_text(
    query="hardware useful for running machine learning models and CUDA workloads",
    target_vector="description_vector",
    limit=3,
    return_properties=["title", "description", "component"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Component:", obj.properties["component"])
    print("Description:", obj.properties["description"])
    print("-" * 80)

Distance: 0.3125572204589844
Title: NVIDIA RTX 4000 Ada
Component: GPU
Description: A professional GPU with large VRAM, useful for machine learning, rendering, CUDA workloads and local AI experiments.
--------------------------------------------------------------------------------
Distance: 0.4179983139038086
Title: Samsung 990 PRO NVMe SSD
Component: Storage
Description: A very fast NVMe storage drive useful for operating systems, datasets, Docker images, virtual machines and large project files.
--------------------------------------------------------------------------------
Distance: 0.4191930294036865
Title: AMD Ryzen 9 Processor
Component: CPU
Description: A high core count CPU useful for compiling code, virtualization, multitasking and running development environments.
--------------------------------------------------------------------------------


In [10]:
query = "fast disk for datasets and virtual machines"

print("SEARCH BY TITLE VECTOR")
response = hardware.query.near_text(
    query=query,
    target_vector="title_vector",
    limit=3,
    return_properties=["title", "component", "use_case"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print(obj.properties)
    print("-" * 80)


print("\nSEARCH BY DESCRIPTION VECTOR")
response = hardware.query.near_text(
    query=query,
    target_vector="description_vector",
    limit=3,
    return_properties=["title", "component", "use_case"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print(obj.properties)
    print("-" * 80)

SEARCH BY TITLE VECTOR
Distance: 0.5939450263977051
{'component': 'Storage', 'title': 'Samsung 990 PRO NVMe SSD', 'use_case': 'fast storage'}
--------------------------------------------------------------------------------
Distance: 0.620046854019165
{'component': 'RAM', 'title': '64 GB DDR5 RAM Kit', 'use_case': 'memory intensive workloads'}
--------------------------------------------------------------------------------
Distance: 0.6890332102775574
{'component': 'CPU', 'title': 'AMD Ryzen 9 Processor', 'use_case': 'development workstation'}
--------------------------------------------------------------------------------

SEARCH BY DESCRIPTION VECTOR
Distance: 0.41405022144317627
{'component': 'Storage', 'title': 'Samsung 990 PRO NVMe SSD', 'use_case': 'fast storage'}
--------------------------------------------------------------------------------
Distance: 0.5161846876144409
{'component': 'RAM', 'title': '64 GB DDR5 RAM Kit', 'use_case': 'memory intensive workloads'}
----------------

In [11]:
from weaviate.classes.query import Filter

response = hardware.query.near_text(
    query="hardware for AI and machine learning",
    target_vector="description_vector",
    filters=Filter.by_property("component").equal("GPU"),
    limit=3,
    return_properties=["title", "description", "component"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print(obj.properties)
    print("-" * 80)

Distance: 0.3444627523422241
{'component': 'GPU', 'title': 'NVIDIA RTX 4000 Ada', 'description': 'A professional GPU with large VRAM, useful for machine learning, rendering, CUDA workloads and local AI experiments.'}
--------------------------------------------------------------------------------


In [12]:
def search_hardware(query: str, *, target_vector: str = "description_vector", limit: int = 3) -> None:
    response = hardware.query.near_text(
        query=query,
        target_vector=target_vector,
        limit=limit,
        return_properties=["title", "component", "use_case", "description"],
        return_metadata=MetadataQuery(distance=True),
    )

    print(f"Query: {query}")
    print(f"Target vector: {target_vector}")
    print("-" * 80)

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Title:", obj.properties["title"])
        print("Component:", obj.properties["component"])
        print("Use case:", obj.properties["use_case"])
        print("Description:", obj.properties["description"])
        print("-" * 80)

In [13]:
search_hardware("AI model training and CUDA workloads")

Query: AI model training and CUDA workloads
Target vector: description_vector
--------------------------------------------------------------------------------
Distance: 0.4598628282546997
Title: NVIDIA RTX 4000 Ada
Component: GPU
Use case: AI workstation
Description: A professional GPU with large VRAM, useful for machine learning, rendering, CUDA workloads and local AI experiments.
--------------------------------------------------------------------------------
Distance: 0.6182266473770142
Title: AMD Ryzen 9 Processor
Component: CPU
Use case: development workstation
Description: A high core count CPU useful for compiling code, virtualization, multitasking and running development environments.
--------------------------------------------------------------------------------
Distance: 0.6247899532318115
Title: 64 GB DDR5 RAM Kit
Component: RAM
Use case: memory intensive workloads
Description: Large system memory helps with Docker containers, virtual machines, data science notebooks and la

In [14]:
search_hardware("fast storage for Docker images and datasets")

Query: fast storage for Docker images and datasets
Target vector: description_vector
--------------------------------------------------------------------------------
Distance: 0.48289042711257935
Title: Samsung 990 PRO NVMe SSD
Component: Storage
Use case: fast storage
Description: A very fast NVMe storage drive useful for operating systems, datasets, Docker images, virtual machines and large project files.
--------------------------------------------------------------------------------
Distance: 0.5034129619598389
Title: 64 GB DDR5 RAM Kit
Component: RAM
Use case: memory intensive workloads
Description: Large system memory helps with Docker containers, virtual machines, data science notebooks and large datasets.
--------------------------------------------------------------------------------
Distance: 0.6836572885513306
Title: 850W Modular Power Supply
Component: PSU
Use case: workstation power
Description: A reliable PSU provides stable power for workstations with powerful GPUs, CPUs

In [15]:
search_hardware("many cores for compiling code and virtualization")

Query: many cores for compiling code and virtualization
Target vector: description_vector
--------------------------------------------------------------------------------
Distance: 0.31241506338119507
Title: AMD Ryzen 9 Processor
Component: CPU
Use case: development workstation
Description: A high core count CPU useful for compiling code, virtualization, multitasking and running development environments.
--------------------------------------------------------------------------------
Distance: 0.564774215221405
Title: 64 GB DDR5 RAM Kit
Component: RAM
Use case: memory intensive workloads
Description: Large system memory helps with Docker containers, virtual machines, data science notebooks and large datasets.
--------------------------------------------------------------------------------
Distance: 0.6501055955886841
Title: Samsung 990 PRO NVMe SSD
Component: Storage
Use case: fast storage
Description: A very fast NVMe storage drive useful for operating systems, datasets, Docker images

In [16]:
search_hardware(
    "NVIDIA graphics card",
    target_vector="title_vector",
)

Query: NVIDIA graphics card
Target vector: title_vector
--------------------------------------------------------------------------------
Distance: 0.42690855264663696
Title: NVIDIA RTX 4000 Ada
Component: GPU
Use case: AI workstation
Description: A professional GPU with large VRAM, useful for machine learning, rendering, CUDA workloads and local AI experiments.
--------------------------------------------------------------------------------
Distance: 0.5365774035453796
Title: Samsung 990 PRO NVMe SSD
Component: Storage
Use case: fast storage
Description: A very fast NVMe storage drive useful for operating systems, datasets, Docker images, virtual machines and large project files.
--------------------------------------------------------------------------------
Distance: 0.5603469014167786
Title: PCIe 4.0 Motherboard
Component: Motherboard
Use case: hardware expansion
Description: A motherboard with modern chipset features, PCIe slots, fast storage connectors and support for future hardw

In [17]:
client.close()